JobHop is a career trajectory dataset, so the cleanest formulation is:
- CV = a person’s past experiences (sequence of ESCO occupations)
- JD = the next occupation they moved into (represented using your ESCO JD text from jd_df)

This becomes a ranking problem:
- Given a CV history, rank candidate job descriptions; the true next job should rank highly.

<a class="anchor" id="chapter1"></a>

# 1. Imports

</a>

In [2]:
from datasets import load_dataset

/opt/anaconda3/envs/ResumeMatcherThesis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
import numpy as np

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [4]:
ds = load_dataset("aida-ugent/JobHop")

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 1341832
    })
    validation: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 167945
    })
    test: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 167924
    })
})

In [ ]:
train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

In [ ]:
train_df = train_ds.to_pandas()
val_df = val_ds.to_pandas()
test_df = test_ds.to_pandas()

<a class="anchor" id="chapter3"></a>

# 3. Initial Analysis

</a>

<a class="anchor" id="sub-section-3_1"></a>

## 3.1. Types & Structure

</a>

In [9]:
print("Shape of train df:", train_df.shape)
train_df.head()

Shape of train df: (1341832, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,0,resource manager,Resource managers manage resources for all pot...,1324.8.3,Q1 2016,Q2 2019,True
1,0,health and safety officer,Health and safety officers execute plans for t...,2263.3,Q1 2017,Q2 2019,True
2,0,integration engineer,Integration engineers develop and implement so...,2511.17,Q1 2013,Q1 2016,True
3,0,programme manager,Programme managers coordinate and oversee seve...,1213.4,Q2 2012,Q1 2013,True
4,0,product development engineering drafter,Product development engineering drafters desig...,3118.3.12,Q1 2011,Q2 2012,True


In [10]:
print("Shape of validation df:", val_df.shape)
val_df.head()

Shape of validation df: (167945, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,5,medical administrative assistant,Medical administrative assistants work very cl...,3344.1,Q1 2013,Q4 2013,True
1,5,administrative assistant,Administrative assistants provide administrati...,3343.1,Q4 2013,Q4 2018,True
2,5,room attendant,"Room attendants clean, tidy and restock guest ...",9112.4,None,None,True
3,6,unknown,unknown,None,Q1 2010,Q4 2011,True
4,6,unknown,unknown,None,Q1 2009,None,True


In [11]:
print("Shape of test df:", test_df.shape)
test_df.head()

Shape of test df: (167924, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,16,psychology lecturer,"Psychology lecturers are subject professors, t...",2310.1.35,Q1 2012,Q2 2012,True
1,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q1 2009,Q3 2011,True
2,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q4 2003,Q4 2008,True
3,16,human resources manager,"Human resources managers plan, design and impl...",1212.2,Q3 2000,Q4 2003,True
4,36,leaflet distributor,"Leaflet distributors hand out flyers, leaflet ...",9510.1,Q1 2005,Q4 2006,False


In [12]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341832 entries, 0 to 1341831
Data columns (total 7 columns):
 #   Column               Non-Null Count    Dtype 
---  ------               --------------    ----- 
 0   person_id            1341832 non-null  int64 
 1   matched_label        1341832 non-null  object
 2   matched_description  1341832 non-null  object
 3   matched_code         1330799 non-null  object
 4   start_date           1187091 non-null  object
 5   end_date             1043416 non-null  object
 6   university_studies   1341832 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 62.7+ MB


In [13]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167945 entries, 0 to 167944
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167945 non-null  int64 
 1   matched_label        167945 non-null  object
 2   matched_description  167945 non-null  object
 3   matched_code         166430 non-null  object
 4   start_date           148683 non-null  object
 5   end_date             130359 non-null  object
 6   university_studies   167945 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


In [14]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167924 entries, 0 to 167923
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167924 non-null  int64 
 1   matched_label        167924 non-null  object
 2   matched_description  167924 non-null  object
 3   matched_code         166596 non-null  object
 4   start_date           149029 non-null  object
 5   end_date             131111 non-null  object
 6   university_studies   167924 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


<a class="anchor" id="sub-section-3_2"></a>

## 3.2. Duplicates

</a>

In [17]:
print(train_df[train_df.index.duplicated() == True])
print(val_df[val_df.index.duplicated() == True])
print(test_df[test_df.index.duplicated() == True])

Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []


In [34]:
print("Train: ", train_df.duplicated().mean())
print("Validation: ", val_df.duplicated().mean())
print("Test: ", test_df.duplicated().mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [38]:
train_duplicates = train_df.duplicated().sum()
val_duplicates = val_df.duplicated().sum()
test_duplicates = test_df.duplicated().sum()
print(f"There are {train_duplicates} duplicated rows in train_df.")
print(f"There are {val_duplicates} duplicated rows in val_df.")
print(f"There are {test_duplicates} duplicated rows in test_df.")

There are 35340 duplicated rows in train_df.
There are 4463 duplicated rows in val_df.
There are 4307 duplicated rows in test_df.


In [36]:
print("Train: ", train_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Validation: ", val_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Test: ", test_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [39]:
train_df = train_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
val_df = val_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
test_df = test_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])

<a class="anchor" id="sub-section-3_3"></a>

## 3.3. Missing Values

</a>

In [22]:
train_df.isna().sum()

person_id                   0
matched_label               0
matched_description         0
matched_code            11033
start_date             154741
end_date               298416
university_studies          0
dtype: int64

In [23]:
val_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1515
start_date             19262
end_date               37586
university_studies         0
dtype: int64

In [24]:
test_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1328
start_date             18895
end_date               36813
university_studies         0
dtype: int64

<a class="anchor" id="sub-section-3_4"></a>

## 3.4. Statistical Analysis

</a>

In [26]:
train_df.describe(include = object).T

,count,unique,top,freq
matched_label,1341832,2955,unknown,217294
matched_description,1341832,2955,unknown,217294
matched_code,1330799,2955,unknown,206261
start_date,1187091,227,Q1 2011,62721
end_date,1043416,216,Q4 2012,59725


In [27]:
val_df.describe(include = object).T

,count,unique,top,freq
matched_label,167945,2606,unknown,27357
matched_description,167945,2606,unknown,27357
matched_code,166430,2606,unknown,25842
start_date,148683,203,Q1 2011,7820
end_date,130359,190,Q4 2012,7488


In [28]:
test_df.describe(include = object).T

,count,unique,top,freq
matched_label,167924,2619,unknown,27186
matched_description,167924,2619,unknown,27186
matched_code,166596,2619,unknown,25858
start_date,149029,203,Q1 2011,7739
end_date,131111,194,Q4 2013,7408


In [43]:
print("Unknown rate for train: ", (train_df['matched_code'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_code'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_code'] == 'unknown').mean())

Unknown rate for train:  0.15364885510205956
Unknown rate for validation:  0.15386403396092535
Unknown rate for test:  0.15390821247180916


These rows do not have a valid ESCO occupation mapping. The original resume text could not be aligned to any ESCO occupation, therefore: no occupation, no ESCO skills, no ontology relations. These rows are structurally unusable for ESCO-augmented CV–JD matching.

In [44]:
train_df = train_df[train_df['matched_code'] != 'unknown']
val_df = val_df[val_df['matched_code'] != 'unknown']
test_df = test_df[test_df['matched_code'] != 'unknown']

In [46]:
print("Train shape: ", train_df.shape)
train_df.describe(include = object).T

Train shape:  (1105751, 7)


,count,unique,top,freq
matched_label,1105751,2955,administrative assistant,56538
matched_description,1105751,2955,Administrative assistants provide administrati...,56538
matched_code,1095528,2954,3343.1,56538
start_date,971522,224,Q1 2011,51374
end_date,830844,212,Q4 2012,46961


<a class="anchor" id="chapter4"></a>

# 4. CV

</a>

<a class="anchor" id="sub-section-4_1"></a>

## 4.1. Check

</a>

**Check people with less than 2 experiences**

In [48]:
train_df.groupby('person_id')['matched_code'].nunique().describe()

count    278149.000000
mean          3.312829
std           1.920530
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          18.000000
Name: matched_code, dtype: float64

In [51]:
print("People with ≥ 2 occupations in train: ", (train_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in val: ", (val_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in test: ", (test_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())

People with ≥ 2 occupations in train:  0.8260320907139699
People with ≥ 2 occupations in val:  0.8257521335593805
People with ≥ 2 occupations in test:  0.8262920249633315


For modeling and evaluation, the sample should be restricted to individuals with at least two distinct occupations, as career transitions are required to construct CV–job pairs. Individuals with a single recorded occupation were retained for descriptive analysis but excluded from supervised matching experiments.

In [60]:
def valid_persons(df):
    return df.groupby('person_id')['matched_code'].nunique().loc[lambda x: x >= 2].index

train_df = train_df[train_df['person_id'].isin(valid_persons(train_df))]
val_df = val_df[val_df['person_id'].isin(valid_persons(val_df))]
test_df = test_df[test_df['person_id'].isin(valid_persons(test_df))]

In [61]:
#train_df.groupby('person_id')['matched_code'].nunique().describe()
#val_df.groupby('person_id')['matched_code'].nunique().describe()
test_df.groupby('person_id')['matched_code'].nunique().describe()

count    28731.000000
mean         3.814869
std          1.759112
min          2.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         16.000000
Name: matched_code, dtype: float64

**Check date consistency**

In [59]:
def quarter_to_timestamp(q):
    if pd.isna(q):
        return np.nan
    try:
        quarter, year = q.split()
        year = int(year)
        q_num = int(quarter[1])
        month = (q_num - 1) * 3 + 1
        return pd.Timestamp(year=year, month=month, day=1)
    except:
        return np.nan

In [69]:
train_df['start_ts'] = train_df['start_date'].apply(quarter_to_timestamp)
train_df['end_ts'] = train_df['end_date'].apply(quarter_to_timestamp)

val_df['start_ts'] = val_df['start_date'].apply(quarter_to_timestamp)
val_df['end_ts'] = val_df['end_date'].apply(quarter_to_timestamp)

test_df['start_ts'] = test_df['start_date'].apply(quarter_to_timestamp)
test_df['end_ts'] = test_df['end_date'].apply(quarter_to_timestamp)

In [67]:
train_df[['start_date', 'end_date', 'start_ts', 'end_ts']].sample(10)

,start_date,end_date,start_ts,end_ts
829040,Q1 2008,None,2008-01-01,NaT
59228,Q1 2010,Q4 2011,2010-01-01,2011-10-01
627189,Q2 2015,Q3 2015,2015-04-01,2015-07-01
642430,Q1 1990,Q4 2002,1990-01-01,2002-10-01
505782,Q1 2003,Q4 2004,2003-01-01,2004-10-01
333251,Q1 1992,Q4 1995,1992-01-01,1995-10-01
1142151,Q1 2017,Q4 2017,2017-01-01,2017-10-01
1038533,Q1 1996,Q4 2014,1996-01-01,2014-10-01
765263,Q1 1985,Q4 1986,1985-01-01,1986-10-01
903738,Q1 2013,Q4 2015,2013-01-01,2015-10-01


In [70]:
print("Invalid rate train: ", (train_df['start_ts'] > train_df['end_ts']).mean())
print("Invalid rate val: ", (val_df['start_ts'] > val_df['end_ts']).mean())
print("Invalid rate test: ", (test_df['start_ts'] > test_df['end_ts']).mean())

Invalid rate train:  1.6352458976970927e-05
Invalid rate val:  1.5383077076907694e-05
Invalid rate test:  1.5364051193018575e-05


Quarter-level dates are converted to timestamps for temporal ordering. Temporal inconsistencies affect fewer than 0.01% of records across all splits and can be tolerated. Career sequences should be ordered by start date, with end date used as a fallback.

<a class="anchor" id="sub-section-4_2"></a>

## 4.2. Constructing CVs

</a>

In [21]:
cv_train = (
    train_df
    .groupby("person_id")
    .agg({
        "matched_code": list,
        "matched_label": list,
        "matched_description": list,
        "university_studies": "max"
    })
    .reset_index()
)

In [22]:
cv_train.head()

,person_id,matched_code,matched_label,matched_description,university_studies
0,0,"[1324.8.3, 2263.3, 2511.17, 1213.4, 3118.3.12,...","[resource manager, health and safety officer, ...",[Resource managers manage resources for all po...,True
1,1,"[2512.2, 2512.2, 2153.1.1, unknown, 5414.1.9, ...","[software analyst, software analyst, telecommu...",[Software analysts elicit and prioritise user ...,True
2,2,"[2411.1, 2411.1, 2433.6.6, 4110.1]","[accountant, accountant, technical sales repre...",[Accountants review and analyse financial stat...,True
3,4,[2611.1],[lawyer],[Lawyers provide legal advice to clients and a...,False
4,8,"[unknown, unknown, unknown, unknown]","[unknown, unknown, unknown, unknown]","[unknown, unknown, unknown, unknown]",False


In [23]:
len(cv_train)

288965

Each résumé is constructed by aggregating all job experiences associated with a unique person identifier, yielding a structured CV representation grounded in ESCO occupation descriptions.